# Projeto de Reconhecimento de Gestos com MobileNetV2 e GRU

Este notebook consolida um projeto de reconhecimento de gestos baseado em vídeo. O fluxo de trabalho é dividido em duas etapas principais:

1.  **Extração de Características (CNN)**: Utiliza uma rede MobileNetV2 pré-treinada para extrair um vetor de características de cada quadro dos vídeos.
2.  **Classificação de Sequências (RNN)**: Alimenta as sequências de características extraídas em uma rede GRU (Gated Recurrent Unit) para classificar o gesto em cada vídeo.

## Estrutura do Notebook

1.  **Configuração**: Instalação de dependências e importações.
2.  **Definições**: Definição de constantes, paths, loggers, funções utilitárias e classes de modelo (CNN e GRU).
3.  **Preparação de Dados**: Instruções e código para configurar os dados necessários (vídeos e anotações).
4.  **Execução Principal**: Células separadas para executar a extração de características, o treinamento do modelo GRU e os testes de escalabilidade.

## 1. Configuração Inicial e Dependências

In [ ]:
# Célula 1: Instalar dependências
# A biblioteca InquirerPy foi removida pois não é compatível com notebooks.
!pip install torch torchvision pandas tqdm opencv-python-headless seaborn matplotlib -q

In [ ]:
# Célula 2: Importações Globais

#---------- biblioteca padrão ----------
import re
import os
import time
from pathlib import Path
import logging
from typing import Final

#---------- biblioteca de terceiros ----------
import cv2
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # Usar tqdm.notebook para melhor exibição no Colab
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from sklearn.metrics import confusion_matrix, classification_report

## 2. Definições de Constantes, Funções e Classes

In [ ]:
# Célula 3: Constantes e Configuração de Paths (adaptado de const.py)

# Base do projeto no Colab
ROOT: Final = Path('/content/')

# Diretórios
MODELS_PATH: Final = ROOT / "saved_models"
CNN_PATH: Final = MODELS_PATH / "cnn_model.pt"
RNN_PATH: Final = MODELS_PATH / "gru_model.pt"
VIDEOS_PATH: Final = ROOT / "videos"
FEATURES_PATH: Final = ROOT / "features_train"
FEATURES_VAL_PATH: Final = ROOT / "features_val"
FEATURES_TESTE_PATH: Final = ROOT / "features_teste"
LOGS_PATH: Final = ROOT / "logs"
TEST_PATH: Final = ROOT / "test"

# CSVs
TEST_CSV_PATH = TEST_PATH / "csv_test/"
ONE_THREAD_CSV:Final = TEST_CSV_PATH / "80_videos.csv"
TWO_THREAD_CSV:Final = TEST_CSV_PATH / "160_videos.csv"
FOUR_THREAD_CSV:Final = TEST_CSV_PATH / "320_videos.csv"
EIGHT_THREAD_CSV:Final = TEST_CSV_PATH / "640_videos.csv"
CSV_PATH: Final = VIDEOS_PATH / "annotations.csv"
FEATURES_CSV_PATH: Final = FEATURES_PATH / "annotations.csv"
FEATURES_CSV_VAL_PATH: Final = FEATURES_VAL_PATH / "annotations.csv"
FEATURES_CSV_TESTE_PATH: Final = FEATURES_TESTE_PATH / "annotations.csv"

# Logs e gráficos
LOG_CNN: Final = "logCNN.log"
LOG_RNN: Final = "logRNN.log"
LOG_ESC_STRONG: Final = "escalabilidade_forte.csv"
LOG_ESC_WEAK: Final = "escalabilidade_fraca.csv"
LOG_CNN_PATH: Final = LOGS_PATH / LOG_CNN
LOG_RNN_PATH: Final = LOGS_PATH / LOG_RNN
TEST_ESC_STRONG: Final = TEST_PATH / LOG_ESC_STRONG
TEST_ESC_WEAK: Final = TEST_PATH / LOG_ESC_WEAK
RNN_GRAPH_PATH: Final = LOGS_PATH / "graficoRNN.png"
RNN_MATRIX_PATH: Final = LOGS_PATH / "matrizRNN.png"

#---------- Configuração de Redes e Threads ----------
BATCH_SIZE = 4
NUM_WORKERS = 2 # Pode aumentar no Colab Pro
PIN_MEMORY = True # Recomendado quando se usa GPU
CUDA = True # Usar CUDA se disponível
CUDA_VALUE = "cuda"
CPU_VALUE = "cpu"

# Criar diretórios necessários
for path in [MODELS_PATH, VIDEOS_PATH, FEATURES_PATH, FEATURES_VAL_PATH, FEATURES_TESTE_PATH, LOGS_PATH, TEST_PATH, TEST_CSV_PATH]:
    os.makedirs(path, exist_ok=True)

print("Diretórios criados:")
print(f"- Vídeos: {VIDEOS_PATH}")
print(f"- Features: {FEATURES_PATH}, {FEATURES_VAL_PATH}, {FEATURES_TESTE_PATH}")
print(f"- Modelos Salvos: {MODELS_PATH}")

In [ ]:
# Célula 4: Logger (de logger.py)

def setup_logger(path, namefile):
    # Garante que a pasta exista
    os.makedirs(os.path.dirname(path), exist_ok=True)

    logger = logging.getLogger(namefile)
    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(logging.DEBUG)
    
    # Formato consistente
    fmt = logging.Formatter("%(message)s")

    # Console: só INFO+
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # Arquivo: DEBUG+
    fh = logging.FileHandler(path)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    
    logger.propagate = False
    return logger

loggerCNN = setup_logger(LOG_CNN_PATH, LOG_CNN)
loggerRNN = setup_logger(LOG_RNN_PATH, LOG_RNN)

print(f"Log da CNN será salvo em: {LOG_CNN_PATH}")
print(f"Log do RNN será salvo em: {LOG_RNN_PATH}")

In [ ]:
# Célula 5: Funções Utilitárias (de utils.py)

def save_model(model, label_map, save_dir=MODELS_PATH, model_name='model'):
    os.makedirs(save_dir, exist_ok=True)
    model_scripted = torch.jit.script(model.cpu().eval())
    
    checkpoint_path = os.path.join(save_dir, f"{model_name}_checkpoint.pth")
    model_path = os.path.join(save_dir, f"{model_name}.pt")

    torch.save({
        'model_state_dict': model.state_dict(),
        'label_map': label_map
    }, checkpoint_path)
    
    model_scripted.save(model_path)
    print(f"Modelo '{model_name}' salvo em: {save_dir}")

def plot_training_curves(train_losses, val_losses, train_accs, val_accs):
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.title('Curvas de Perda (Loss)')
    plt.xlabel('Épocas')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(val_accs, label='Val Accuracy')
    plt.title('Curvas de Acurácia')
    plt.xlabel('Épocas')
    plt.ylabel('Acurácia (%)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(RNN_GRAPH_PATH)
    plt.show()

def create_feature_annotations_csv(path_features, csv_path):
    name_features = sorted([f for f in os.listdir(path_features) if f.endswith('.pt')])
    if not name_features:
        #print(f"Atenção: Nenhum arquivo .pt encontrado em {path_features}")
        return
    class_name = []
    for cl in name_features:
        match = re.search('-Sinalizador', cl)
        if match:
            class_name.append(cl[:match.start()])
        else:
            class_name.append('unknown')

    df = pd.DataFrame({
        "video_name": [name.replace('.mp4', '.pt') for name in name_features],
        "class": class_name,
    })
    df.to_csv(csv_path, index=False)
    #print(f"CSV de anotações de features criado em: {csv_path}")

def plot_conf_matrix(model, dataloader, dataset, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sequences, labels, lengths in tqdm(dataloader, desc="Gerando matriz de confusão"):
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences, lengths)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    class_names = list(dataset.idx2label.values())
    cm = confusion_matrix(all_labels, all_preds, labels=list(dataset.idx2label.keys()))
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(15, 12))
    sns.heatmap(cm_normalized, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão Normalizada')
    plt.xlabel('Classe Predita')
    plt.ylabel('Classe Verdadeira')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(RNN_MATRIX_PATH)
    plt.show()

In [ ]:
# Célula 6: Código de Extração de Features (de cnn.py)

class CNNDataset(Dataset):
    def __init__(self, annotations_file, videosDir, transform=None):
        self.annotations_file = Path(annotations_file)
        if not self.annotations_file.exists():
            raise FileNotFoundError(f"Arquivo de anotações não encontrado: {self.annotations_file}")
        df = pd.read_csv(self.annotations_file)
        self.labels_df = df[["class"]]
        self.videos_name_df = df[["video_name"]]
        
        classes_em_ordem = sorted(list(self.labels_df["class"].unique()))
        self.label2idx = {classe: i for i, classe in enumerate(classes_em_ordem)}
        self.idx2label = {i: classe for classe, i in self.label2idx.items()}
        
        self.videosDir = Path(videosDir)
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        video_name = self.videos_name_df.iloc[idx, 0]
        video_path = self.videosDir / video_name
        video_frames = self.extractFrames(str(video_path))
        label = self.label2idx[self.labels_df.iloc[idx, 0]]

        if not video_frames:
            # Retorna um tensor vazio se o vídeo não puder ser lido
            return torch.empty(0), label, video_name

        frames_t = [self.transform(frame) for frame in video_frames]
        videoT = torch.stack(frames_t, dim=0)
        
        return videoT, label, video_name

    def extractFrames(self, filepath):
        cap = cv2.VideoCapture(filepath)
        frames = []
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        return frames

    @staticmethod
    def collate_fn(batch):
        # Filtra vídeos que falharam ao carregar
        batch = [item for item in batch if item[0].numel() > 0]
        if not batch:
            return None, None, None, None
            
        sequences, labels, video_names = zip(*batch)
        lengths = [seq.shape[0] for seq in sequences]
        padded_sequences = pad_sequence(sequences=sequences, batch_first=True, padding_value=0)
        labels = torch.tensor(labels, dtype=torch.long)
        return padded_sequences, labels, video_names, lengths


class CNNMobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        for p in backbone.parameters():
            p.requires_grad = False
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return x

def extract_features(dataloader: DataLoader, device, video_id_teste=3, video_id_val=5):
    extractor = CNNMobileNetV2().to(device=device)
    extractor.eval()
    loggerCNN.info("batch_id,runtime,vps,fps")
    print("Iniciando extração de features usando MobileNetV2")
    print('-' * 50)
    
    with torch.no_grad():
        for batch_idx, (videos, labels, names, lengths) in enumerate(tqdm(dataloader, desc="Extraindo batches")):
            if videos is None: # Batch vazio devido a erro no collate_fn
                continue

            start_time = time.time()
            try:
                B, T, C, H, W = videos.shape
                flat_videos = videos.view(B * T, C, H, W).to(device)
                feats_flats = extractor(flat_videos)
                D = feats_flats.size(1)
                feats = feats_flats.view(B, T, D).cpu()
                
                for i in range(B):
                    video_name = names[i]
                    video_name_no_ext = video_name.replace(".mp4", "")
                    real_T = lengths[i]
                    
                    match = re.search('-Sinalizador(\d+)', video_name_no_ext)
                    if match and match.group(1).isdigit():
                        chave = int(match.group(1))
                        if chave % video_id_val == 0:
                            out_path = FEATURES_VAL_PATH / f"{video_name_no_ext}.pt"
                        elif chave % video_id_teste == 0:
                            out_path = FEATURES_TESTE_PATH / f"{video_name_no_ext}.pt"
                        else:
                             out_path = FEATURES_PATH / f"{video_name_no_ext}.pt"
                    else:
                        out_path = FEATURES_PATH / f"{video_name_no_ext}.pt"
                        
                    torch.save({
                        'features': feats[i, :real_T, :], # Salva apenas os frames reais
                        'length': real_T
                    }, out_path)

                elapsed = time.time() - start_time
                videos_per_sec = B / elapsed if elapsed > 0 else 0
                frames_per_sec = sum(lengths) / elapsed if elapsed > 0 else 0
                loggerCNN.info(f"{batch_idx},{elapsed:.2f},{videos_per_sec:.1f},{frames_per_sec:.1f}")
            except Exception as e:
                loggerCNN.error(f"Erro no batch {batch_idx}: {e}", exc_info=True)
                print(f"Erro no batch {batch_idx}: {e}")

    print("\nCriando arquivos CSV para os conjuntos de features...")
    create_feature_annotations_csv(FEATURES_PATH, FEATURES_CSV_PATH)
    create_feature_annotations_csv(FEATURES_TESTE_PATH, FEATURES_CSV_TESTE_PATH)
    create_feature_annotations_csv(FEATURES_VAL_PATH, FEATURES_CSV_VAL_PATH)
    return extractor

In [ ]:
# Célula 7: Código do Modelo GRU (de rnn.py)

# Parâmetros do modelo GRU
INPUT_DIM = 1280 
HIDDEN_DIM = 512
NUM_LAYERS = 2
NUM_CLASSES = 20 # ATENÇÃO: Este valor será atualizado dinamicamente
DROPOUT = 0.4
NUM_EPOCHS = 80
LEARNING_RATE = 0.001

class RNNDataset(Dataset):
    def __init__(self, annotations_file, featuresDir, label2idx=None):
        self.annotations_file = Path(annotations_file)
        if not self.annotations_file.exists() or os.path.getsize(self.annotations_file) == 0:
            # print(f"Arquivo de anotações vazio ou não encontrado: {self.annotations_file}")
            self.df = pd.DataFrame(columns=["video_name", "class"])
        else:
             self.df = pd.read_csv(self.annotations_file)

        if label2idx is None:
            classes = sorted(list(self.df["class"].unique()))
            self.label2idx = {c: i for i, c in enumerate(classes)}
        else:
            self.label2idx = label2idx
        self.idx2label = {v: k for k, v in self.label2idx.items()}
        
        self.featuresDir = Path(featuresDir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        feature_filename = self.df.iloc[idx, 0]
        feature_path = self.featuresDir / feature_filename
        feature_data = torch.load(feature_path, map_location=torch.device('cpu'))
        
        features = feature_data['features']
        label_str = self.df.iloc[idx, 1]
        label = self.label2idx[label_str]
        
        return features, label

    @staticmethod
    def rnn_collate_fn(batch):
        sequences, labels = zip(*batch)
        lengths = [seq.shape[0] for seq in sequences]
        padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
        labels = torch.tensor(labels, dtype=torch.long)
        return padded_sequences, labels, lengths

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        # Assegurar que lengths esteja na CPU para pack_padded_sequence
        cpu_lengths = [l for l in lengths] 
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths=cpu_lengths, batch_first=True, enforce_sorted=False
        )
        _output, hidden = self.gru(packed)
        last_layer_hidden = hidden[-1, :, :]
        out = self.dropout(last_layer_hidden)
        out = self.fc(out)
        return out

def run_training_loop(device, model, train_loader, val_loader, num_epochs=15):
    runtimeCode = time.time()
    loggerRNN.info("epoch,runtime,train_loss,train_acc,val_loss,val_acc")
    print("Iniciando treinamento do modelo GRU")
    print('-' * 50)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=5, verbose=True)
    
    train_losses, train_accs, val_losses, val_accs = [], [], [], []

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        # Treino
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for sequences, labels, lengths in tqdm(train_loader, desc=f'Época {epoch+1}/{num_epochs} [Treino]', leave=False):
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences, lengths)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * sequences.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        epoch_train_loss = running_loss / total
        epoch_train_acc = 100 * correct / total
        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc)

        # Validação
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        if val_loader and len(val_loader) > 0:
            with torch.no_grad():
                for sequences, labels, lengths in tqdm(val_loader, desc=f'Época {epoch+1}/{num_epochs} [Validação]', leave=False):
                    sequences, labels = sequences.to(device), labels.to(device)
                    outputs = model(sequences, lengths)
                    loss = criterion(outputs, labels)
                    val_running_loss += loss.item() * sequences.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
            epoch_val_loss = val_running_loss / val_total
            epoch_val_acc = 100 * val_correct / val_total
            scheduler.step(epoch_val_loss)
        else:
            epoch_val_loss, epoch_val_acc = 0.0, 0.0
        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc)

        epoch_runtime = time.time() - epoch_start_time
        loggerRNN.info(f'{epoch+1},{epoch_runtime:.2f},{epoch_train_loss:.4f},{epoch_train_acc:.2f},{epoch_val_loss:.4f},{epoch_val_acc:.2f}')
        print(f'Época {epoch+1:02d} | Tempo: {epoch_runtime:.2f}s | ' 
              f'Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | ' 
              f'Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%')

    total_runtime = time.time() - runtimeCode
    loggerRNN.info(f"Tempo total de execução: {total_runtime:.2f}s")
    print(f'Treinamento concluído em {total_runtime // 60:.0f}m {total_runtime % 60:.0f}s')
    return model, train_losses, train_accs, val_losses, val_accs

## 3. Preparação dos Dados (Ação Necessária)

Antes de prosseguir, você precisa fazer o upload dos seus dados.

1.  No painel à esquerda, clique no ícone de pasta para abrir o explorador de arquivos.
2.  Navegue até a pasta `/content/videos/`.
3.  **Faça o upload do seu arquivo `annotations.csv` e de todos os seus vídeos `.mp4` para esta pasta.**

O arquivo `annotations.csv` deve ter pelo menos duas colunas: `video_name` e `class`.

---

In [ ]:
# Célula 8: Criar arquivos de exemplo (execute se não tiver seus próprios dados)
# Este código cria um arquivo annotations.csv e vídeos de exemplo para evitar erros.
# SUBSTITUA PELOS SEUS DADOS REAIS.

print("Criando dados de exemplo...")

# Criar um CSV de exemplo
sample_data = {
    'video_name': [
        'Aplaudir-Sinalizador1.mp4',
        'Tchau-Sinalizador6.mp4', # Val
        'Ok-Sinalizador5.mp4',    # Teste
        'Aplaudir-Sinalizador2.mp4',
        'Tchau-Sinalizador3.mp4', # Teste
        'Ok-Sinalizador10.mp4'    # Val
    ],
    'class': ['Aplaudir', 'Tchau', 'Ok', 'Aplaudir', 'Tchau', 'Ok']
}
sample_df = pd.DataFrame(sample_data)
sample_df.to_csv(CSV_PATH, index=False)
print(f"Arquivo de anotações de exemplo criado em: {CSV_PATH}")
print(sample_df.to_string())

# Criar vídeos de exemplo (vídeos pretos curtos)
for video_name in sample_data['video_name']:
    video_path = VIDEOS_PATH / video_name
    width, height = 224, 224
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(video_path), fourcc, 10.0, (width, height))
    if not out.isOpened():
        print(f"Erro ao abrir o VideoWriter para {video_path}")
        continue
    for _ in range(np.random.randint(20, 40)): # Duração aleatória
        frame = np.random.randint(0, 255, (height, width, 3), dtype=np.uint8)
        out.write(frame)
    out.release()
print(f"\nVídeos de exemplo criados em {VIDEOS_PATH}")

## 4. Execução Principal

### 4.1. Extrair Features com a CNN (MobileNetV2)

In [ ]:
# Célula 9: Função de inicialização e execução da CNN

def run_feature_extraction():
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    try:
        dataset = CNNDataset(
            annotations_file=CSV_PATH,
            videosDir=VIDEOS_PATH,
            transform=transform
        )
    except FileNotFoundError as e:
        print(f"Erro: {e}")
        print("Por favor, certifique-se de que seus dados estão na pasta /content/videos/ e execute a célula de preparação de dados.")
        return

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=CNNDataset.collate_fn
    )
    
    if CUDA and torch.cuda.is_available():
        device = torch.device(CUDA_VALUE)
        print(f"Usando dispositivo: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device(CPU_VALUE)
        print("Usando dispositivo: CPU")

    cnn_model = extract_features(dataloader=loader, device=device)
    save_model(model=cnn_model, label_map=dataset.idx2label, model_name='cnn_feature_extractor')

# Execute esta célula para iniciar a extração de características
run_feature_extraction()

### 4.2. Treinar o Classificador (GRU)

In [ ]:
# Célula 10: Função de inicialização e execução do GRU

def run_gru_training():
    try:
        dataset_train = RNNDataset(
            annotations_file=FEATURES_CSV_PATH,
            featuresDir=FEATURES_PATH
        )
    except Exception as e:
        print(f"Erro ao carregar dataset de treino: {e}")
        print("Verifique se a extração de features foi executada e gerou arquivos em .pt e anotações .csv")
        return
    
    if len(dataset_train) == 0:
        print("Dataset de treino está vazio. Abortando treinamento.")
        return
    
    label_map = dataset_train.label2idx
    global NUM_CLASSES
    NUM_CLASSES = len(label_map)
    print(f"Encontradas {NUM_CLASSES} classes no dataset de treino.")

    loader_train = DataLoader(
        dataset_train, batch_size=BATCH_SIZE, shuffle=True, pin_memory=PIN_MEMORY,
        collate_fn=RNNDataset.rnn_collate_fn, num_workers=NUM_WORKERS
    )
    dataset_val = RNNDataset(FEATURES_CSV_VAL_PATH, FEATURES_VAL_PATH, label2idx=label_map)
    loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, collate_fn=RNNDataset.rnn_collate_fn) if len(dataset_val) > 0 else None

    dataset_test = RNNDataset(FEATURES_CSV_TESTE_PATH, FEATURES_TESTE_PATH, label2idx=label_map)
    loader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, collate_fn=RNNDataset.rnn_collate_fn) if len(dataset_test) > 0 else None

    device = torch.device(CUDA_VALUE if CUDA and torch.cuda.is_available() else CPU_VALUE)
    print(f"Usando dispositivo: {device}")

    gru_model = GRUModel(
        input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
        num_classes=NUM_CLASSES, dropout=DROPOUT
    ).to(device)

    model_trained, train_loss, train_acc, val_loss, val_acc = run_training_loop(
        device=device, model=gru_model, train_loader=loader_train,
        val_loader=loader_val, num_epochs=NUM_EPOCHS
    )
    
    save_model(model=model_trained, label_map=dataset_train.idx2label, model_name='gru_classifier')
    plot_training_curves(train_loss, val_loss, train_acc, val_acc)
    
    if loader_test:
        print("\nGerando matriz de confusão no conjunto de teste...")
        plot_conf_matrix(model=model_trained, dataloader=loader_test, dataset=dataset_test, device=device)
    else:
        print("\nConjunto de teste vazio. Matriz de confusão não será gerada.")

# Execute esta célula para iniciar o treinamento do GRU
run_gru_training()

### 4.3. Testes de Escalabilidade

In [ ]:
# Célula 11: Funções de Teste de Escalabilidade (de cnn.py)

def feature_extraction_for_test(dataloader: DataLoader, device):
    extractor = CNNMobileNetV2().to(device=device)
    extractor.eval()
    with torch.no_grad():
        for batch in dataloader:
            videos, _, _, _ = batch
            if videos is None: continue
            B, T, C, H, W = videos.shape
            flat = videos.view(B * T, C, H, W).to(device)
            _ = extractor(flat)

def run_scaling_test_execution(num_thread, annotations_csv_path=CSV_PATH):
    torch.set_num_threads(num_thread)
    device = torch.device('cpu')
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((112, 112)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    dataset = CNNDataset(annotations_file=annotations_csv_path, videosDir=VIDEOS_PATH, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, collate_fn=CNNDataset.collate_fn)
    
    start_time = time.time()
    feature_extraction_for_test(dataloader=loader, device=device)
    return time.time() - start_time

def run_strong_scaling_test():
    torch.set_num_interop_threads(1)
    results = []
    threads_list = [1, 2, 4, 8]
    print("Iniciando teste de escalabilidade forte (carga de trabalho fixa)")
    print('-' * 60)
    base_time = 0
    for n in threads_list:
        print(f"Executando com num_threads = {n}")
        total_time_strong = run_scaling_test_execution(n)
        if n == 1:
            base_time = total_time_strong
            speedup, eff = 1.0, 100.0
        else:
            speedup = base_time / total_time_strong if total_time_strong > 0 else float('inf')
            eff = (speedup / n) * 100
        results.append({
            "threads": n, "total_time": total_time_strong,
            "speedup": round(speedup, 2), "efficiency (%)": round(eff, 1),
        })
    df = pd.DataFrame(results)
    df.to_csv(TEST_ESC_STRONG, index=False)
    print("\nResultados do Teste de Escalabilidade Forte:")
    print(df.to_string())
    print(f"\nResultados salvos em {TEST_ESC_STRONG}")

def run_weak_scaling_test():
    threads_list = [1, 2, 4, 8]
    paths = [ONE_THREAD_CSV, TWO_THREAD_CSV, FOUR_THREAD_CSV, EIGHT_THREAD_CSV]
    
    # Criar os arquivos CSV para o teste fraco
    try:
        base_df = pd.read_csv(CSV_PATH)
        if len(base_df) == 0: raise ValueError("CSV base está vazio.")
    except (FileNotFoundError, ValueError) as e:
        print(f"Erro ao ler CSV base para teste de escalabilidade fraca: {e}")
        return
        
    for n, path in zip(threads_list, paths):
        num_repeats = n
        weak_df = pd.concat([base_df] * num_repeats, ignore_index=True)
        weak_df.to_csv(path, index=False)
    
    results = []
    print("\nIniciando teste de escalabilidade fraca (carga de trabalho crescente)")
    print('-' * 60)
    base_time = 0
    for n_thread, csv_path in zip(threads_list, paths):
        print(f"Executando com num_threads = {n_thread} e tamanho da base = {len(pd.read_csv(csv_path))}")
        total_time_weak = run_scaling_test_execution(n_thread, csv_path)
        if n_thread == 1:
            base_time = total_time_weak
            speedup, eff = 1.0, 100.0
        else:
            # No teste fraco, speedup ideal é 1 (tempo se mantém constante)
            speedup = base_time / total_time_weak if total_time_weak > 0 else float('inf')
            eff = speedup * 100
        results.append({
            "threads": n_thread, "total_time": total_time_weak,
            "speedup": round(speedup, 2), "efficiency (%)": round(eff, 1),
        })
    df = pd.DataFrame(results)
    df.to_csv(TEST_ESC_WEAK, index=False)
    print("\nResultados do Teste de Escalabilidade Fraca:")
    print(df.to_string())
    print(f"\nResultados salvos em {TEST_ESC_WEAK}")

In [ ]:
# Célula 12: Executar os testes de escalabilidade (Opcional)

print("Atenção: Os testes de escalabilidade de CPU no Colab podem não refletir o desempenho em hardware dedicado, pois os recursos de CPU são compartilhados.")
print("Descomente as linhas abaixo para executar.")

# run_strong_scaling_test()
# print("\n" + "="*60 + "\n")
# run_weak_scaling_test()